In [10]:
"""
Document Ingestion Pipeline for GenAI/RAG
==========================================
Routes files by extension to the appropriate loader, extracts text/tables,
and normalizes everything into a common chunk schema ready for embedding.
"""

import os
import uuid
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional


# ---------------------------------------------------------------------------
# Common chunk schema — every loader below produces a list of these
# ---------------------------------------------------------------------------

@dataclass
class Chunk:
    id: str
    source: str            # file path
    content: str            # normalized text
    chunk_type: str          # "text", "table", "code", "image_ocr"
    metadata: dict = field(default_factory=dict)  # page num, sheet name, etc.


def make_chunk(source: str, content: str, chunk_type: str, **metadata) -> Chunk:
    return Chunk(
        id=str(uuid.uuid4()),
        source=source,
        content=content.strip(),
        chunk_type=chunk_type,
        metadata=metadata,
    )


# ---------------------------------------------------------------------------
# PDF loader — PyMuPDF for text, pdfplumber for tables
# ---------------------------------------------------------------------------

def load_pdf(path: str) -> list[Chunk]:
    import fitz
    import pdfplumber

    chunks = []

    # Text extraction (fast, page-by-page)
    doc = fitz.open(path)
    for page_num, page in enumerate(doc):
        text = page.get_text()
        if text.strip():
            chunks.append(make_chunk(path, text, "text", page=page_num + 1))
    doc.close()

    # Table extraction (more accurate via pdfplumber)
    with pdfplumber.open(path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            for table in page.extract_tables():
                table_text = table_to_markdown(table)
                if table_text:
                    chunks.append(make_chunk(path, table_text, "table", page=page_num + 1))

    return chunks


def table_to_markdown(table: list[list]) -> str:
    """Convert a list-of-lists table into a markdown table string."""
    if not table or not table[0]:
        return ""
    rows = [[cell or "" for cell in row] for row in table]
    header, *body = rows
    md = "| " + " | ".join(header) + " |\n"
    md += "| " + " | ".join(["---"] * len(header)) + " |\n"
    for row in body:
        md += "| " + " | ".join(row) + " |\n"
    return md


# Fallback for scanned/image-only PDFs: OCR each page
def load_pdf_ocr(path: str) -> list[Chunk]:
    import fitz
    import pytesseract
    from PIL import Image
    import io

    chunks = []
    doc = fitz.open(path)
    for page_num, page in enumerate(doc):
        pix = page.get_pixmap(dpi=300)
        img = Image.open(io.BytesIO(pix.tobytes("png")))
        text = pytesseract.image_to_string(img)
        if text.strip():
            chunks.append(make_chunk(path, text, "image_ocr", page=page_num + 1))
    doc.close()
    return chunks


def is_scanned_pdf(path: str, sample_pages: int = 3) -> bool:
    """Heuristic: if extracted text is near-empty across sample pages, treat as scanned."""
    import fitz
    doc = fitz.open(path)
    total_chars = 0
    for page in doc[:sample_pages]:
        total_chars += len(page.get_text().strip())
    doc.close()
    return total_chars < 20  # essentially no extractable text


# ---------------------------------------------------------------------------
# Word document loader
# ---------------------------------------------------------------------------

def load_docx(path: str) -> list[Chunk]:
    from docx import Document

    doc = Document(path)
    chunks = []

    buffer = []
    for para in doc.paragraphs:
        if para.style.name.startswith("Heading") and buffer:
            chunks.append(make_chunk(path, "\n".join(buffer), "text"))
            buffer = []
        if para.text.strip():
            buffer.append(para.text)
    if buffer:
        chunks.append(make_chunk(path, "\n".join(buffer), "text"))

    for i, table in enumerate(doc.tables):
        rows = [[cell.text for cell in row.cells] for row in table.rows]
        chunks.append(make_chunk(path, table_to_markdown(rows), "table", table_index=i))

    return chunks


# ---------------------------------------------------------------------------
# HTML loader
# ---------------------------------------------------------------------------

def load_html(path: str) -> list[Chunk]:
    from bs4 import BeautifulSoup

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    # Strip non-content tags
    for tag in soup(["script", "style", "nav", "footer"]):
        tag.decompose()

    chunks = []
    for el in soup.find_all(["p", "li", "h1", "h2", "h3", "table"]):
        if el.name == "table":
            rows = [[c.get_text(strip=True) for c in row.find_all(["td", "th"])]
                    for row in el.find_all("tr")]
            chunks.append(make_chunk(path, table_to_markdown(rows), "table"))
        else:
            text = el.get_text(strip=True)
            if text:
                chunks.append(make_chunk(path, text, "text", tag=el.name))

    return chunks


# ---------------------------------------------------------------------------
# CSV / Excel loader
# ---------------------------------------------------------------------------

def load_tabular(path: str, rows_per_chunk: int = 20) -> list[Chunk]:
    import pandas as pd

    ext = Path(path).suffix.lower()
    sheets = {"Sheet1": pd.read_csv(path)} if ext == ".csv" else pd.read_excel(path, sheet_name=None)

    chunks = []
    for sheet_name, df in sheets.items():
        for start in range(0, len(df), rows_per_chunk):
            block = df.iloc[start:start + rows_per_chunk]
            chunks.append(make_chunk(
                path, block.to_markdown(index=False), "table",
                sheet=sheet_name, rows=f"{start}-{start + len(block)}"
            ))
    return chunks


# ---------------------------------------------------------------------------
# Code loader — tree-sitter, chunked by function/class
# ---------------------------------------------------------------------------

def load_code(path: str) -> list[Chunk]:
    from tree_sitter import Language, Parser
    import tree_sitter_python as tspython  # swap per language

    parser = Parser(Language(tspython.language()))

    with open(path, "rb") as f:
        code_bytes = f.read()

    tree = parser.parse(code_bytes)
    chunks = []

    def walk(node):
        if node.type in ("function_definition", "class_definition"):
            snippet = code_bytes[node.start_byte:node.end_byte].decode(errors="ignore")
            chunks.append(make_chunk(
                path, snippet, "code",
                node_type=node.type, start_line=node.start_point[0] + 1
            ))
            return  # don't descend into nested defs separately
        for child in node.children:
            walk(child)

    walk(tree.root_node)

    if not chunks:  # fallback: no functions/classes found, chunk whole file
        chunks.append(make_chunk(path, code_bytes.decode(errors="ignore"), "code"))

    return chunks


# ---------------------------------------------------------------------------
# Router
# ---------------------------------------------------------------------------

LOADERS = {
    ".pdf": load_pdf,
    ".docx": load_docx,
    ".html": load_html,
    ".htm": load_html,
    ".csv": load_tabular,
    ".xlsx": load_tabular,
    ".py": load_code,
}


def load_document(path: str) -> list[Chunk]:
    ext = Path(path).suffix.lower()
    loader = LOADERS.get(ext)
    if not loader:
        raise ValueError(f"No loader registered for extension: {ext}")

    # Special case: route scanned PDFs to OCR
    if ext == ".pdf" and is_scanned_pdf(path):
        return load_pdf_ocr(path)

    return loader(path)


def load_directory(directory: str) -> list[Chunk]:
    """Walk a directory and load every supported file into one chunk list."""
    all_chunks = []
    for root, _, files in os.walk(directory):
        for fname in files:
            fpath = os.path.join(root, fname)
            ext = Path(fpath).suffix.lower()
            if ext in LOADERS:
                try:
                    all_chunks.extend(load_document(fpath))
                except Exception as e:
                    print(f"Failed to load {fpath}: {e}")
    return all_chunks


# ---------------------------------------------------------------------------
# Example usage
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    chunks = load_directory("./downloads")
    print(f"Loaded {len(chunks)} chunks")

    for chunk in chunks[:5]:
        print(f"\n[{chunk.chunk_type}] {chunk.source} | {chunk.metadata}")
        print(chunk.content[:200])

    # Ready to embed, e.g.:
    # embeddings = embed_model.encode([c.content for c in chunks])

Loaded 0 chunks
